# SDXL dual-GPU image generation

Run the Go backend locally (`./run.sh` from repo root). It tunnels a WebSocket URL into the injection cell below and pushes this notebook to Kaggle.

This notebook loads **Stable Diffusion XL** on **2× T4 GPUs**, connects to your machine, and generates two images per prompt in parallel.

If Kaggle does not select GPU T4 x2 automatically, open **Settings → Accelerator → GPU T4 x2**, save a version, then re-push.

In [ ]:
# Injection template
BACKEND_WS_URL = ""

In [ ]:
# Injection space


In [ ]:
!pip install -q diffusers transformers accelerate safetensors websockets

In [ ]:
import asyncio
import io
import random
from concurrent.futures import ThreadPoolExecutor

import torch
import websockets
from diffusers import StableDiffusionXLPipeline
from PIL import Image

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"


def load_pipeline(device):
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe = pipe.to(device)
    pipe.vae.enable_slicing()
    return pipe


def generate_image(pipe, device, prompt, seed):
    generator = torch.Generator(device=device).manual_seed(seed)
    image = pipe(prompt=prompt, generator=generator).images[0]
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()


def generate_pair(pipes, prompt):
    seeds = [random.randint(0, 2**32 - 1) for _ in range(2)]

    def run(gpu_id):
        device = f"cuda:{gpu_id}"
        png = generate_image(pipes[gpu_id], device, prompt, seeds[gpu_id])
        return gpu_id, png

    with ThreadPoolExecutor(max_workers=2) as pool:
        return list(pool.map(run, [0, 1]))


async def generate_with_keepalive(ws, pipes, prompt):
    stop = asyncio.Event()

    async def keepalive():
        while not stop.is_set():
            await asyncio.sleep(25)
            if stop.is_set():
                break
            await ws.send("KEEPALIVE")

    task = asyncio.create_task(keepalive())
    try:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, generate_pair, pipes, prompt)
    finally:
        stop.set()
        task.cancel()


async def run():
    if not BACKEND_WS_URL:
        raise RuntimeError("BACKEND_WS_URL is not set")

    print("Loading SDXL on cuda:0...")
    pipe0 = load_pipeline("cuda:0")
    print("Loading SDXL on cuda:1...")
    pipe1 = load_pipeline("cuda:1")
    pipes = [pipe0, pipe1]

    async with websockets.connect(
        BACKEND_WS_URL,
        max_size=None,
        ping_interval=30,
        ping_timeout=300,
    ) as ws:
        await ws.send("READY")
        print("Connected to backend")

        while True:
            msg = await ws.recv()
            if msg == "DONE":
                break

            print(f"Generating: {msg!r}")
            images = await generate_with_keepalive(ws, pipes, msg)
            for gpu_id, png in sorted(images):
                await ws.send(f"IMG:{gpu_id}")
                await ws.send(png)
            await ws.send("END")
            print("Sent images")


await run()